# Sales Prediction — MLflow + Prefect Pipeline

This notebook builds an end-to-end ML pipeline:

**Data → Validation → Preprocessing → Training → Evaluation → MLflow Tracking → Prefect Orchestration**

> Expected project structure:
> `sales-prediction-mlops/data/raw/sales_data.csv`

Run the cells from top to bottom.


## 1. Install required packages

In [ ]:
# Run this cell once in your active .venv environment.
%pip install -U pandas numpy scikit-learn matplotlib seaborn joblib mlflow prefect

## 2. Imports and project paths

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import mlflow
import mlflow.sklearn

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sales_data.csv"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())


## 3. Load dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place sales_data.csv inside data/raw/ or change DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.info())


## 4. Basic data inspection

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes)

display(df.describe(include="all").T)


## 5. Select the target column

The notebook first tries common target names. If your dataset uses another name, **change `TARGET_COLUMN` manually**.

For a sales dataset, examples may be `Sales`, `sales`, `Revenue`, `Profit`, etc.


In [ ]:
# Change this manually if automatic detection chooses the wrong column.
TARGET_COLUMN = None

common_targets = [
    "target", "Target",
    "sales", "Sales",
    "revenue", "Revenue",
    "profit", "Profit",
    "result", "Result",
    "label", "Label"
]

if TARGET_COLUMN is None:
    matches = [c for c in common_targets if c in df.columns]
    if matches:
        TARGET_COLUMN = matches[0]

if TARGET_COLUMN is None:
    print("Could not automatically determine the target.")
    print("Available columns:", df.columns.tolist())
    raise ValueError("Set TARGET_COLUMN in the previous cell.")

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"{TARGET_COLUMN!r} is not a column in the dataset.")

print("Target column:", TARGET_COLUMN)
print("Target dtype:", df[TARGET_COLUMN].dtype)
print("Unique target values:", df[TARGET_COLUMN].nunique())


## 6. Prepare features and target

In [ ]:
data = df.dropna(subset=[TARGET_COLUMN]).copy()

X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN]

# Remove columns that contain a unique value for every row (usually IDs).
high_cardinality = [
    c for c in X.columns
    if X[c].nunique(dropna=False) == len(X)
]

if high_cardinality:
    print("Dropping ID-like columns:", high_cardinality)
    X = X.drop(columns=high_cardinality)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Features:", X.columns.tolist())
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


## 7. Build preprocessing pipeline

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessor created.")


## 8. Detect problem type and split data

In [ ]:
# Heuristic:
# relatively small number of distinct non-numeric labels -> classification;
# numeric target with many unique values -> regression.

if (
    y.dtype == "object"
    or str(y.dtype).startswith("category")
    or y.nunique() <= 20
):
    PROBLEM_TYPE = "classification"
else:
    PROBLEM_TYPE = "regression"

stratify = y if PROBLEM_TYPE == "classification" and y.nunique() > 1 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=stratify
)

print("Problem type:", PROBLEM_TYPE)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 9. Train model

In [ ]:
if PROBLEM_TYPE == "classification":
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
else:
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)
print("Model training completed.")


## 10. Evaluate model

In [ ]:
predictions = pipeline.predict(X_test)

metrics = {}

if PROBLEM_TYPE == "classification":
    metrics["accuracy"] = float(accuracy_score(y_test, predictions))

    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, predictions))

    cm = confusion_matrix(y_test, predictions)
    print("Confusion matrix:")
    print(cm)

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

else:
    metrics["mae"] = float(mean_absolute_error(y_test, predictions))
    metrics["rmse"] = float(np.sqrt(mean_squared_error(y_test, predictions)))
    metrics["r2"] = float(r2_score(y_test, predictions))

    print(f"MAE : {metrics['mae']:.4f}")
    print(f"RMSE: {metrics['rmse']:.4f}")
    print(f"R²  : {metrics['r2']:.4f}")

    plt.figure(figsize=(7, 5))
    plt.scatter(y_test, predictions, alpha=0.7)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted")
    plt.show()

print("\nMetrics:", metrics)


## 11. Save model and metrics

In [ ]:
MODEL_PATH = MODEL_DIR / "sales_prediction_model.pkl"
METRICS_PATH = REPORT_DIR / "metrics.json"

joblib.dump(pipeline, MODEL_PATH)

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=4)

print("Model saved:", MODEL_PATH)
print("Metrics saved:", METRICS_PATH)


## 12. Start MLflow tracking

For a local MLflow server, open a **separate terminal** in the project directory and run:

```text
mlflow server --host 127.0.0.1 --port 5000
```

Then open:

`http://127.0.0.1:5000`

The notebook below logs the experiment to a local `mlruns` directory. This works without a database server and is useful for learning.


In [ ]:
MLFLOW_EXPERIMENT = "sales-prediction"

mlflow.set_tracking_uri("file://" + str((PROJECT_ROOT / "mlruns").resolve()))
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", MLFLOW_EXPERIMENT)


## 13. Log training run to MLflow

In [ ]:
with mlflow.start_run(run_name="random-forest-sales-model") as run:
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("problem_type", PROBLEM_TYPE)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("target_column", TARGET_COLUMN)
    mlflow.log_param("training_records", len(X_train))
    mlflow.log_param("testing_records", len(X_test))

    mlflow.log_metrics(metrics)

    mlflow.log_artifact(str(METRICS_PATH))
    mlflow.log_artifact(str(MODEL_PATH))

    # Log the sklearn pipeline as an MLflow model.
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model"
    )

    RUN_ID = run.info.run_id

print("MLflow run completed.")
print("Run ID:", RUN_ID)


## 14. Check MLflow experiment

In [ ]:
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)

print("Experiment ID:", experiment.experiment_id if experiment else None)

if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    display(runs[[
        c for c in [
            "run_id", "status", "metrics.accuracy",
            "metrics.mae", "metrics.rmse", "metrics.r2",
            "params.model_type", "params.problem_type"
        ] if c in runs.columns
    ]])


# 15. Prefect pipeline

Now we convert the important steps into Prefect tasks.

Prefect will orchestrate:

`load data → train model → evaluate → MLflow log`

This is the key difference:

- **Prefect = workflow orchestration**
- **MLflow = experiment/model tracking**


In [ ]:
from prefect import flow, task

@task(name="load-data")
def load_data_task(path: str):
    data = pd.read_csv(path)
    return data

@task(name="train-model")
def train_model_task(data: pd.DataFrame):
    X = data.drop(columns=[TARGET_COLUMN])
    y = data[TARGET_COLUMN]

    # Remove ID-like columns
    id_cols = [c for c in X.columns if X[c].nunique(dropna=False) == len(X)]
    X = X.drop(columns=id_cols)

    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(exclude=["number"]).columns.tolist()

    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])

    if y.dtype == "object" or str(y.dtype).startswith("category") or y.nunique() <= 20:
        problem_type = "classification"
        estimator = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        stratify = y if y.nunique() > 1 else None
    else:
        problem_type = "regression"
        estimator = RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
        stratify = None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=stratify
    )

    pipe = Pipeline([
        ("preprocessor", prep),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)

    return pipe, X_test, y_test, problem_type

@task(name="evaluate-model")
def evaluate_model_task(pipe, X_test, y_test, problem_type):
    pred = pipe.predict(X_test)

    if problem_type == "classification":
        result = {
            "accuracy": float(accuracy_score(y_test, pred))
        }
    else:
        result = {
            "mae": float(mean_absolute_error(y_test, pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
            "r2": float(r2_score(y_test, pred))
        }

    return result

@task(name="log-to-mlflow")
def log_mlflow_task(pipe, metrics_result, problem_type):
    mlflow.set_tracking_uri(
        "file://" + str((PROJECT_ROOT / "mlruns").resolve())
    )
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="prefect-random-forest-run") as run:
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("problem_type", problem_type)
        mlflow.log_param("target_column", TARGET_COLUMN)
        mlflow.log_param("n_estimators", 200)
        mlflow.log_metrics(metrics_result)

        mlflow.sklearn.log_model(
            pipe,
            artifact_path="model"
        )

        return run.info.run_id

@flow(name="sales-mlops-pipeline")
def sales_mlops_flow():
    data = load_data_task(str(DATA_PATH))
    pipe, X_test, y_test, problem_type = train_model_task(data)
    metrics_result = evaluate_model_task(pipe, X_test, y_test, problem_type)
    run_id = log_mlflow_task(pipe, metrics_result, problem_type)

    print("Pipeline completed.")
    print("Metrics:", metrics_result)
    print("MLflow Run ID:", run_id)

    return metrics_result

print("Prefect flow created.")


## 16. Run the complete Prefect pipeline

In [ ]:
result = sales_mlops_flow()
print(result)


## 17. Start Prefect UI

Open a **separate terminal**:

```text
prefect server start
```

Then open the Prefect dashboard shown by the command, normally:

`http://127.0.0.1:4200`

The flow should appear as:

`SALES-MLOPS-PIPELINE`

Run the notebook pipeline again to create a fresh flow run.


## 18. Final project structure

After running the notebook, you should have:

```text
sales-prediction-mlops/
│
├── data/
│   └── raw/
│       └── sales_data.csv
│
├── models/
│   └── sales_prediction_model.pkl
│
├── reports/
│   └── metrics.json
│
├── mlruns/
│   └── MLflow experiment data
│
└── notebook/
    └── sales_mlflow_prefect_pipeline.ipynb
```

### Final architecture

```text
                    ┌──────────────┐
                    │ sales_data   │
                    │    .csv      │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │   Prefect    │
                    │ Orchestration│
                    └──────┬───────┘
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
          Load Data    Train Model   Evaluate
              │            │            │
              └────────────┼────────────┘
                           ▼
                    ┌──────────────┐
                    │    MLflow    │
                    ├──────────────┤
                    │ Parameters   │
                    │ Metrics      │
                    │ Model        │
                    │ Artifacts    │
                    └──────────────┘
```

**Important:** If the automatic target detection selects the wrong column, return to Section 5 and set `TARGET_COLUMN` explicitly.
